# Transformers from scratch · Part 3: fine-tuning a pretrained GPT-2

In Part 2 we trained a GPT **from scratch**: it knew nothing, and everything it learned came from one million characters of Shakespeare. Real LLMs are used the other way round. A big model is **pre-trained** once on a huge amount of general text, and then **fine-tuned**, trained a little more on the data you care about. This notebook does that with GPT-2 and compares it with Part 2's tiny GPT.

| | |
|---|---|
| **Task** | the same as Part 2: predict the next token of Shakespeare |
| **Data** | Tiny Shakespeare, with the same train/validation split as Part 2 |
| **Model** | GPT-2 small (OpenAI, 2019): 124 million parameters, pre-trained on 40 GB of web text |
| **Metric** | validation loss in **bits per character**, so it compares fairly with Part 2's character-level model |

**Steps**

1. Load the pre-trained GPT-2 and its tokenizer
2. Subword tokens (BPE): how GPT-2 cuts up text
3. Before fine-tuning: how well does GPT-2 already predict Shakespeare?
4. Fine-tuning
5. After fine-tuning: the comparison, in numbers and in samples
6. What changed, and why it works

It needs the Hugging Face `transformers` library (`pip install transformers`) and downloads GPT-2 (about 500 MB) the first time. Fine-tuning takes a few minutes on an RTX 4090.

In [ ]:
# Settings
MODEL_NAME = "openai-community/gpt2"   # GPT-2 small, 124M parameters
BLOCK_SIZE = 256      # tokens per training sequence
BATCH_SIZE = 8        # sequences per step
MAX_STEPS = 400       # fine-tuning steps (a few passes over the training text)
LR = 5e-5             # peak learning rate: much smaller than when training from scratch
WARMUP = 40
EVAL_EVERY = 50

## 0 · Setup

In [ ]:
import contextlib, copy, math, time, urllib.request
from pathlib import Path

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = True
autocast = (lambda: torch.autocast("cuda", dtype=torch.bfloat16)) if device == "cuda" else contextlib.nullcontext
print("PyTorch", torch.__version__, "| device:", device)

path = Path("data/tinyshakespeare.txt")
if not path.exists():
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", path)
text = path.read_text(encoding="utf-8")
n = int(0.9 * len(text))
train_text, val_text = text[:n], text[n:]          # the same split as Part 2
print(f"train {len(train_text):,} characters | validation {len(val_text):,} characters")

## 1 · Load the pre-trained GPT-2

`from_pretrained` downloads the weights OpenAI published in 2019 and builds the model around them. It's the same kind of model as Part 2's: token and position embeddings, 12 pre-norm blocks of causal self-attention + MLP, a final LayerNorm and a tied output layer. It's just bigger, and already trained.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
cfg = model.config
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters | {cfg.n_layer} blocks | {cfg.n_head} heads | "
      f"d_model {cfg.n_embd} | context {cfg.n_positions} tokens | vocabulary {cfg.vocab_size:,} tokens")
pretrained_state = copy.deepcopy(model.state_dict())     # a copy of the original weights, to compare with later

## 2 · Subword tokens (BPE)

GPT-2's tokenizer uses **byte-pair encoding** (BPE). It starts from single bytes and repeatedly merges the most frequent adjacent pair into a new token, 50,000 times, on a large text corpus. Frequent words become a single token, and rare words are split into a few common pieces, so nothing is ever unknown. `Ġ` marks a token that starts with a space.

Compared with Part 2's characters, sequences get about 3–4 times shorter, so the same 256-token context covers about 3–4 times more text.

In [ ]:
sample = "To be, or not to be: that is the question. Thou art a most unkindly villain!"
print(tokenizer.tokenize(sample))
print(tokenizer.encode(sample))

tokenizer.model_max_length = 10**9        # we encode the whole text once; the model only ever sees 256-token windows
train_ids = torch.tensor(tokenizer.encode(train_text))
val_ids = torch.tensor(tokenizer.encode(val_text))
print(f"\ntrain {len(train_ids):,} tokens | validation {len(val_ids):,} tokens | "
      f"{len(val_text) / len(val_ids):.2f} characters per token (Part 2: exactly 1)")

## 3 · Before fine-tuning: how well does GPT-2 already predict Shakespeare?

**Comparing losses across tokenizers.** A loss per token isn't comparable between models that use different tokens: a GPT-2 token covers about 3 characters, a Part 2 token exactly one. So we add up the loss over the whole validation text and divide by the number of **characters**, which gives **bits per character**, the same for any tokenizer.

The evaluation reads the validation text in consecutive chunks of `BLOCK_SIZE` tokens and predicts every token from the earlier tokens of its chunk.

In [ ]:
@torch.no_grad()
def bits_per_char(model, ids, n_chars, block=BLOCK_SIZE):
    # Total negative log-likelihood of the text (in nats) / number of characters / ln 2
    model.eval()
    total = 0.0
    for start in range(0, len(ids) - 1, block):
        chunk = ids[start : start + block + 1][None].to(device)
        with autocast():
            logits = model(chunk[:, :-1]).logits
        total += F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), chunk[:, 1:].reshape(-1), reduction="sum").item()
    return total / n_chars / math.log(2)

@torch.no_grad()
def sample_text(model, prompt="ROMEO:", n_tokens=120, temperature=0.8, top_k=40):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=n_tokens, do_sample=True, temperature=temperature, top_k=top_k,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0])

zero_shot_bpc = bits_per_char(model, val_ids, len(val_text))
print(f"GPT-2 before fine-tuning: {zero_shot_bpc:.3f} bits per character\n")
torch.manual_seed(1)
print(sample_text(model))

GPT-2 already writes fluent English: it learned grammar, common words and a lot of facts during pre-training. But it writes like the web, not like a play. Fine-tuning is meant to keep that general knowledge and adapt the style.

## 4 · Fine-tuning

Fine-tuning is ordinary training (Part 2's loop), started from the pre-trained weights instead of random ones. Three things differ:

- **A much smaller learning rate** (5e-5 here vs 1e-3 in Part 2): we only want to nudge weights that are already good. A large rate would overwrite what pre-training learned, which is called *catastrophic forgetting*.
- **Few steps**: the training text is small, and a 124M-parameter model can memorise it quickly. We measure the validation loss often and keep the best weights.
- **The loss**: Hugging Face models compute the next-token loss themselves when you pass `labels`; they shift the labels by one position internally, exactly as we did by hand in Part 2.

Here every weight of the model is updated (**full fine-tuning**). Section 6 mentions the cheaper alternative used for large models.

In [ ]:
def get_batch(ids):
    starts = torch.randint(len(ids) - BLOCK_SIZE, (BATCH_SIZE,))
    return torch.stack([ids[s : s + BLOCK_SIZE] for s in starts]).to(device)     # (B, T)

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return LR / 10 + (LR - LR / 10) * 0.5 * (1 + math.cos(math.pi * progress))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history, best = [(0, zero_shot_bpc)], (zero_shot_bpc, None)
start = time.time()
model.train()
for step in range(1, MAX_STEPS + 1):
    for group in optimizer.param_groups:
        group["lr"] = lr_at(step - 1)
    batch = get_batch(train_ids)
    with autocast():
        loss = model(batch, labels=batch).loss          # next-token cross-entropy (labels are shifted inside)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % EVAL_EVERY == 0:
        bpc = bits_per_char(model, val_ids, len(val_text))
        history.append((step, bpc))
        if bpc < best[0]:
            best = (bpc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        print(f"step {step:>4} | train loss {loss.item():.3f} nats/token | validation {bpc:.3f} bits/char | {time.time() - start:.0f}s")
        model.train()

if best[1] is not None:
    model.load_state_dict(best[1])
finetuned_bpc = best[0]
print(f"\nbest validation: {finetuned_bpc:.3f} bits per character")

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.plot(*zip(*history), marker="o")
plt.xlabel("fine-tuning step"); plt.ylabel("bits per character"); plt.title("GPT-2 validation loss during fine-tuning")
plt.show()

## 5 · After fine-tuning: the comparison

Three models on the same validation text: Part 2's tiny GPT trained from scratch (loaded from `tiny_gpt.pt` if you ran Part 2 in the same folder), GPT-2 as pre-trained, and GPT-2 after fine-tuning. Below the table are samples from the same prompt.

In [ ]:
# Part 2's tiny GPT, rebuilt from its checkpoint (a compact copy of Part 2's model code, without dropout)
from types import SimpleNamespace
import torch.nn as nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.0):
        super().__init__()
        self.h, self.d_k = n_heads, d_model // n_heads
        self.w_q, self.w_k, self.w_v = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.attn_dropout, self.out_dropout = nn.Dropout(dropout), nn.Dropout(dropout)
        self.register_buffer("causal", torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))

    def forward(self, x):
        B, T, _ = x.shape
        split = lambda t: t.view(B, T, self.h, self.d_k).transpose(1, 2)
        q, k, v = split(self.w_q(x)), split(self.w_k(x)), split(self.w_v(x))
        scores = (q @ k.transpose(-2, -1) / math.sqrt(self.d_k)).masked_fill(~self.causal[:T, :T], float("-inf"))
        out = (scores.softmax(-1) @ v).transpose(1, 2).reshape(B, T, self.h * self.d_k)
        return self.w_o(out)

class MLP(nn.Module):
    def __init__(self, d_model, dropout=0.0):
        super().__init__()
        self.fc, self.proj, self.dropout = nn.Linear(d_model, 4 * d_model), nn.Linear(4 * d_model, d_model), nn.Dropout(dropout)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn, self.mlp = CausalSelfAttention(d_model, n_heads, block_size), MLP(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    # Returns an object with .logits, like a Hugging Face model, so bits_per_char works for both
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size):
        super().__init__()
        self.tok_emb, self.pos_emb = nn.Embedding(vocab_size, d_model), nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(0.0)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, block_size) for _ in range(n_layers)])
        self.ln_f, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(idx.size(1), device=idx.device))
        for block in self.blocks:
            x = block(x)
        return SimpleNamespace(logits=self.head(self.ln_f(x)))

tiny_bpc = None
if Path("tiny_gpt.pt").exists():
    ckpt = torch.load("tiny_gpt.pt", map_location="cpu")
    chars, c = ckpt["chars"], ckpt["config"]
    tiny = TinyGPT(len(chars), c["d_model"], c["n_heads"], c["n_layers"], c["block_size"])
    tiny.load_state_dict(ckpt["model"])
    tiny.to(device).eval()
    stoi = {ch: i for i, ch in enumerate(chars)}
    tiny_val_ids = torch.tensor([stoi[ch] for ch in val_text])
    tiny_bpc = bits_per_char(tiny, tiny_val_ids, len(val_text), block=c["block_size"])
else:
    print("tiny_gpt.pt not found: run Part 2 in this folder to include it in the comparison")

print(f"{'model':<34} {'parameters':>10} {'bits/char':>10}")
if tiny_bpc is not None:
    print(f"{'tiny GPT, from scratch (Part 2)':<34} {sum(p.numel() for p in tiny.parameters()) / 1e6:>9.1f}M {tiny_bpc:>10.3f}")
print(f"{'GPT-2, pre-trained only':<34} {124:>9}M {zero_shot_bpc:>10.3f}")
print(f"{'GPT-2, fine-tuned':<34} {124:>9}M {finetuned_bpc:>10.3f}")

In [ ]:
torch.manual_seed(1)
print("=== GPT-2 after fine-tuning ===")
print(sample_text(model))

original = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
torch.manual_seed(1)
print("\n=== GPT-2 before fine-tuning (the original weights) ===")
print(sample_text(original))
del original

## 6 · What changed, and why it works

**How much did the weights move?** Fine-tuning changes every weight a little, not a few weights a lot. The relative change below is small for every part of the network: the model keeps its pre-trained knowledge and shifts its style.

In [ ]:
changes = {}
for name, w in model.state_dict().items():
    if w.dtype.is_floating_point and w.dim() >= 2 and name in pretrained_state:
        w0 = pretrained_state[name].to(w.device)
        changes[name] = ((w - w0).norm() / w0.norm()).item()
for name, change in sorted(changes.items(), key=lambda kv: -kv[1])[:5]:
    print(f"{name:<45} relative change {change:.2%}")
print(f"... median over all {len(changes)} weight matrices: {sorted(changes.values())[len(changes) // 2]:.2%}")

**Why pre-training helps so much.** Part 2's model had to learn English spelling, grammar and meaning from one million characters. GPT-2 learned those from 40 GB of text before it ever saw Shakespeare, so fine-tuning only has to teach the style: names in capitals before each line, verse, archaic words. This transfer is the reason LLMs are pre-trained once, at great cost, and then adapted cheaply to many tasks. Chat models are made the same way: base models fine-tuned on conversations, then trained further with human feedback.

**Fine-tuning large models cheaply.** Full fine-tuning needs memory for every weight, its gradient and two Adam statistics, about 16 bytes per parameter in mixed precision: 2 GB for GPT-2 small, but more than 100 GB for a 7B model. **LoRA** (low-rank adaptation) freezes the pre-trained weights and trains small extra matrices added next to them, often less than 1% of the parameters, with results close to full fine-tuning. The `peft` library adds it to a Hugging Face model in a few lines.

## 7 · Recap

| Concept | Where it is here |
|---|---|
| pre-training vs fine-tuning (transfer learning) | sections 1, 4, 6 |
| subword tokens (BPE) | section 2 |
| comparing models across tokenizers: bits per character | section 3 |
| a small learning rate, few steps, keeping the best checkpoint | section 4 |
| catastrophic forgetting, LoRA | sections 4 and 6 |

**Experiments to try:**

1. `LR = 1e-3` (Part 2's learning rate): compare the best validation loss with the 5e-5 run. Updates that large overwrite what pre-training learned (catastrophic forgetting).
2. `MAX_STEPS = 2000`: watch the validation loss reach its best and then rise again as the model memorises the training text.
3. Try other prompts in `sample_text`, such as `"JULIET:"` or `"First Citizen:"`, and compare the two models.
4. Train GPT-2's architecture from random weights (`AutoModelForCausalLM.from_config(cfg)`) with the same settings: it ends far behind the fine-tuned model, because it lacks pre-training, not size.